# SmolLM2-recurrent continued pretraining

Upload this notebook to Colab (Runtime > Change runtime type > GPU) and Run All. No editing required.

One-time setup, before the first run: open the key icon in the left sidebar (Secrets) and add a secret named `WANDB_API_KEY` with your Weights & Biases API key (from https://wandb.ai/authorize). If you skip this, training still runs fine -- it just disables wandb logging and prints loss to the cell output instead.

This notebook mounts Google Drive and stores the packed training data and checkpoints there (`MyDrive/smollm2-recurrent/`), so it survives Colab disconnecting you. Every time you reopen this notebook and Run All again, it detects what's already in Drive and picks up where it left off -- skips re-mixing data if it's already packed, and resumes training from the latest saved checkpoint instead of starting over. Free-tier Colab sessions time out well before the full run finishes, so you'll likely need to reopen and Run All several times.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU visible. Go to Runtime > Change runtime type and select a GPU, "
    "then Runtime > Restart session and run this cell again."
)

gpu_name = torch.cuda.get_device_name(0)
major, _minor = torch.cuda.get_device_capability(0)
no_amp = "false" if major >= 8 else "true"
print(f"GPU: {gpu_name} (compute capability {major}.{_minor}) -> no_amp={no_amp}")
print(
    "bf16 autocast enabled." if no_amp == "false" else
    "Pre-Ampere GPU (T4/P100) -- no native bf16, running in fp32 for correctness (slower per-step, but correct)."
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/smollm2-recurrent"
DATA_PATH = f"{DRIVE_ROOT}/data/smollm2_recurrent_mix"
OUT_PATH = f"{DRIVE_ROOT}/huginn_smollm2"
RUN_NAME = "smollm2-recurrent-v1"

import os

os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(OUT_PATH, exist_ok=True)
print(f"Persistent data dir: {DATA_PATH}")
print(f"Persistent checkpoint dir: {OUT_PATH}")

In [ ]:
import os

REPO_DIR = "/content/retrofitting-recurrence"
if not os.path.exists(REPO_DIR):
    !git clone --depth=1 https://github.com/usr-wwelsh/retrofitting-recurrence.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!pip install -q -r requirements.txt

In [ ]:
wandb_disabled = "true"
try:
    from google.colab import userdata

    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    wandb_disabled = "false"
    print("WANDB_API_KEY secret found -- wandb logging enabled.")
except Exception as e:
    print(f"No usable WANDB_API_KEY secret ({e}) -- wandb logging disabled, training still runs fine.")

In [ ]:
import glob

TOKEN_BUDGET = 500_000_000  # ~7,500 steps at batch_size=64 * max_length=1024 below

if glob.glob(f"{DATA_PATH}/*.parquet"):
    print(f"Found existing packed shards in {DATA_PATH}, skipping the mix step.")
else:
    print("No packed data found -- streaming and packing the FineWeb-Edu/DCLM/Cosmopedia-v2 mix.")
    print("This downloads/tokenizes ~500M tokens and can take a while; it resumes cleanly if interrupted (already-written shards are kept, just re-run this cell).")
    !python mix_smollm2_corpus.py --save_path="{DATA_PATH}" --token_budget={TOKEN_BUDGET}

In [ ]:
import re

MAX_STEPS = 7500

ckpt_dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/checkpoint_*")
resume_flag = ""
if ckpt_dirs:
    last_step = max(int(re.search(r"checkpoint_(\d+)", d).group(1)) for d in ckpt_dirs)
    resume_flag = f"--resume_path={OUT_PATH}/{RUN_NAME}/checkpoint_{last_step}"
    print(f"Found checkpoint at step {last_step:,} -- resuming.")
else:
    print("No existing checkpoint -- starting a fresh run.")

!python train.py \
    --run_name={RUN_NAME} \
    --out_path={OUT_PATH} \
    --model_name=usr-wwelsh/Recurrent-SmolLM2-360M-4-14-4 \
    --preprocessed_data_path={DATA_PATH} \
    --is_parquet_dataset=true \
    --max_length=1024 \
    --micro_batch_size=8 \
    --batch_size=64 \
    --optim_config.lr=5e-5 \
    --scheduler_args.warmup=0.02 \
    --scheduler_args.cooldown=0.9 \
    --max_grad_norm=1.0 \
    --no_amp={no_amp} \
    --max_steps={MAX_STEPS} \
    --compile=false \
    --save_interval=250 \
    --wandb_disabled={wandb_disabled} \
    --mean_recurrence_schedule.turn_on=true \
    --mean_recurrence_schedule.warmup=0.25 \
    --mean_recurrence_schedule.max_mean_rec=4 \
    {resume_flag}

## Eval

Only runs to completion if the training cell above finished (reached `MAX_STEPS`) rather than being cut off by a session timeout. Sweeps recurrence depth on the latest checkpoint and compares against base SmolLM2-360M, so you can see whether training is recovering toward (not stuck below) the base model.

In [ ]:
ckpt_dirs = glob.glob(f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_*")
if not ckpt_dirs:
    print("No model_only checkpoint found yet (training hasn't reached a save_interval boundary) -- nothing to eval.")
else:
    last_step = max(int(re.search(r"model_only_chkpt_(\d+)", d).group(1)) for d in ckpt_dirs)
    ckpt_path = f"{OUT_PATH}/{RUN_NAME}/model_only_chkpt_{last_step}"
    print(f"Evaluating {ckpt_path}")

    TASKS = "arc_easy,arc_challenge,hellaswag,mmlu,piqa,winogrande"
    for mean_recurrence in [1, 2, 4, 8]:
        out_dir = f"eval_outputs/{RUN_NAME}/step_{last_step}/mean_recurrence_{mean_recurrence}"
        !lm_eval --model hf \
            --model_args pretrained={ckpt_path},mean_recurrence={mean_recurrence},add_bos_token=True,dtype="float32",trust_remote_code=True \
            --tasks {TASKS} \
            --device cuda \
            --output_path {out_dir} \
            --batch_size auto

    !lm_eval --model hf \
        --model_args pretrained=HuggingFaceTB/SmolLM2-360M,add_bos_token=True,dtype="float32" \
        --tasks {TASKS} \
        --device cuda \
        --output_path eval_outputs/SmolLM2-360M-base \
        --batch_size auto